# QMM Guided JSON Runner

This notebook asks what you need from the run and what you want to simulate.

It then:
1. builds a valid QMM JSON config
2. saves it into `QMM/examples/generated/`
3. runs `qmm`
4. shows the generated outputs

The guided flow uses plain `input()` prompts, so it works without `ipywidgets`.


In [1]:
from pathlib import Path
import json
import sys

candidate_roots = []
seen = set()
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (base, base / 'QMM'):
        resolved = candidate.resolve()
        marker = str(resolved)
        if marker not in seen:
            seen.add(marker)
            candidate_roots.append(resolved)
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'qmm').exists() and (candidate / 'examples').exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise RuntimeError('Could not find the QMM project root from this notebook.')

SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from qmm.qmm_json_helper import (
    PRESET_DESCRIPTIONS,
    available_model_families,
    available_models,
    build_configs,
    choose_python,
    default_request,
    locate_qmm_root,
    parse_float_list,
    parse_yes_no,
    preset_names,
    python_details,
    run_configs,
    save_configs,
    summarize_run,
)

PYTHON_BIN = choose_python(ROOT)
PYTHON_INFO = python_details(ROOT, PYTHON_BIN)
MODELS = available_models(ROOT, PYTHON_BIN)
MODEL_FAMILIES = available_model_families(ROOT, PYTHON_BIN)

print(f'QMM root: {ROOT}')
print(f'Runner Python: {PYTHON_BIN}')
print(f'Runner version: {PYTHON_INFO["version"]}')
print(f'Matplotlib available for plots: {PYTHON_INFO["has_matplotlib"]}')
print('Available mean fields and excluded-volume rules:')
for mean_field, family in MODEL_FAMILIES.items():
    excluded_text = ', '.join(family['excluded_volumes'])
    parameter_text = family['parameter_name'] or 'none'
    print(f"- {mean_field}: excluded_volume={excluded_text}, parameter={parameter_text}")


QMM root: /Users/dajuarez4/Documents/QuarkMatt/QMM
Runner Python: /opt/anaconda3/envs/tovsolver/bin/python
Runner version: [3, 11, 15]
Matplotlib available for plots: True
Available mean fields and excluded-volume rules:
- clausius: excluded_volume=vdw, cs, tvm, parameter=c
- dieterici: excluded_volume=vdw, parameter=alpha
- pr: excluded_volume=vdw, parameter=none
- rks: excluded_volume=vdw, parameter=none
- vdw: excluded_volume=vdw, cs, tvm, parameter=none


In [2]:
print('Available presets:')
for name in preset_names():
    print(f'- {name}: {PRESET_DESCRIPTIONS[name]}')

print('\nAvailable models:')
for name, info in sorted(MODELS.items()):
    parameter = info['parameter_name'] or 'none'
    capabilities = []
    if info['supports_quantum']:
        capabilities.append('quantum')
    if info['supports_symmetric_quarkyonic']:
        capabilities.append('sym-quarkyonic')
    if info['supports_asymmetric_hadronic']:
        capabilities.append('asym-fit')
    if info['supports_asymmetric_quarkyonic']:
        capabilities.append('asym-quarkyonic')
    print(f'- {name}: parameter={parameter}, capabilities={", ".join(capabilities)}')


Available presets:
- ground_state_only: Fit symmetric nuclear matter at saturation only.
- critical_point: Ground state plus the finite-temperature critical point.
- symmetric_quarkyonic: Ground state plus symmetric quarkyonic matter.
- full_symmetric: Ground state, critical point, hadronic EOS, and symmetric quarkyonic matter.
- asymmetric_fixed_y: Asymmetric fixed-y quarkyonic matter without beta equilibrium.
- asymmetric_beta: Asymmetric quarkyonic matter in beta equilibrium.
- asymmetric_beta_neutron_star: Asymmetric beta-equilibrium matter plus a neutron-star sequence.
- baryquark_symmetric: Symmetric baryquark matter demo.
- custom: Choose each workflow block manually.

Available models:
- clausius: parameter=c, capabilities=quantum, sym-quarkyonic, asym-fit, asym-quarkyonic
- clausius_cs: parameter=c, capabilities=quantum, sym-quarkyonic, asym-fit, asym-quarkyonic
- clausius_tvm: parameter=c, capabilities=quantum, sym-quarkyonic, asym-fit, asym-quarkyonic
- cs: parameter=none, c

In [3]:
def choose_option(title, options, default_key=None, descriptions=None):
    keys = list(options)
    print(title)
    for index, key in enumerate(keys, 1):
        label = key
        if descriptions and key in descriptions:
            label = f'{key} - {descriptions[key]}'
        print(f'  {index}. {label}')
    prompt = 'Select option'
    if default_key is not None:
        prompt += f' [{default_key}]'
    prompt += ': '
    raw = input(prompt).strip()
    if not raw:
        if default_key is None:
            raise ValueError('A choice is required.')
        return default_key
    if raw in keys:
        return raw
    if raw.isdigit():
        index = int(raw) - 1
        if 0 <= index < len(keys):
            return keys[index]
    raise ValueError(f'Invalid choice: {raw}')


def workflow_guess(preset, custom_workflows=None):
    workflow = {
        'ground_state': False,
        'quantum_critical': False,
        'hadronic_eos_table': False,
        'symmetric_quarkyonic': False,
        'asymmetric_fit': False,
        'asymmetric_quarkyonic': False,
        'neutron_star': False,
    }
    if preset == 'ground_state_only':
        workflow['ground_state'] = True
    elif preset == 'critical_point':
        workflow['ground_state'] = True
        workflow['quantum_critical'] = True
    elif preset == 'symmetric_quarkyonic':
        workflow['ground_state'] = True
        workflow['symmetric_quarkyonic'] = True
    elif preset == 'full_symmetric':
        workflow['ground_state'] = True
        workflow['quantum_critical'] = True
        workflow['hadronic_eos_table'] = True
        workflow['symmetric_quarkyonic'] = True
    elif preset == 'asymmetric_fixed_y':
        workflow['ground_state'] = True
        workflow['hadronic_eos_table'] = True
        workflow['asymmetric_fit'] = True
        workflow['asymmetric_quarkyonic'] = True
    elif preset == 'asymmetric_beta':
        workflow['ground_state'] = True
        workflow['hadronic_eos_table'] = True
        workflow['asymmetric_fit'] = True
        workflow['asymmetric_quarkyonic'] = True
    elif preset == 'asymmetric_beta_neutron_star':
        workflow['ground_state'] = True
        workflow['hadronic_eos_table'] = True
        workflow['asymmetric_fit'] = True
        workflow['asymmetric_quarkyonic'] = True
        workflow['neutron_star'] = True
    elif preset == 'baryquark_symmetric':
        workflow['ground_state'] = True
        workflow['symmetric_quarkyonic'] = True
    elif preset == 'custom' and custom_workflows is not None:
        workflow.update(custom_workflows)
    return workflow


def prompt_custom_workflows():
    print('Custom workflow selection:')
    keys = [
        'ground_state',
        'quantum_critical',
        'hadronic_eos_table',
        'symmetric_quarkyonic',
        'asymmetric_fit',
        'asymmetric_quarkyonic',
        'neutron_star',
    ]
    values = {}
    for key in keys:
        default = key == 'ground_state'
        values[key] = parse_yes_no(input(f'  Enable {key}? [{"Y" if default else "y/N"}]: '), default)
    return values


def prompt_user_request(models, model_families, python_info):
    base = default_request()
    request = dict(base)

    run_name = input(f'Run name [{base["run_name"]}]: ').strip()
    if run_name:
        request['run_name'] = run_name

    preset = choose_option(
        'What do you want to simulate?',
        preset_names(),
        default_key=base['preset'],
        descriptions=PRESET_DESCRIPTIONS,
    )
    request['preset'] = preset

    mean_field_default = base.get('mean_field', 'vdw')
    mean_field = choose_option(
        'Which mean field do you want to use?',
        sorted(model_families),
        default_key=mean_field_default,
    )
    family = model_families[mean_field]
    excluded_default = base.get('excluded_volume', 'cs') if base.get('excluded_volume') in family['excluded_volumes'] else family['excluded_volumes'][0]
    excluded_volume = choose_option(
        'Which excluded-volume rule do you want to use?',
        family['excluded_volumes'],
        default_key=excluded_default,
    )
    model_name = family['models'][excluded_volume]
    request['mean_field'] = mean_field
    request['excluded_volume'] = excluded_volume
    request['model_name'] = model_name
    model_info = models[model_name]
    print(f'Using QMM model: {model_name}')

    custom_workflows = None
    if preset == 'custom':
        custom_workflows = prompt_custom_workflows()
        request['custom_workflows'] = custom_workflows

    guessed = workflow_guess(preset, custom_workflows)
    needs_quarkyonic = guessed['symmetric_quarkyonic'] or guessed['asymmetric_quarkyonic']
    needs_asymmetric = guessed['asymmetric_fit'] or guessed['asymmetric_quarkyonic'] or guessed['neutron_star']

    if model_info['parameter_name'] is None:
        request['parameter_mode'] = 'none'
        request['parameter_value'] = None
        request['target_k0'] = None
    else:
        parameter_name = model_info['parameter_name']
        parameter_mode = choose_option(
            f'How should {parameter_name} be chosen?',
            ['search', 'value'],
            default_key='search',
        )
        request['parameter_mode'] = parameter_mode
        if parameter_mode == 'search':
            raw_k0 = input(f'Target K0 in MeV [{base["target_k0"]}]: ').strip()
            request['target_k0'] = float(raw_k0) if raw_k0 else float(base['target_k0'])
            request['parameter_value'] = None
        else:
            raw_value = input(f'Explicit value for {parameter_name}: ').strip()
            if not raw_value:
                raise ValueError(f'You must provide a value for {parameter_name}.')
            request['parameter_value'] = float(raw_value)
            request['target_k0'] = None

    request['quick_mode'] = parse_yes_no(
        input('Do you need a fast smoke test with coarser numerics instead of a fuller run? [Y/n]: '),
        True,
    )

    if needs_quarkyonic:
        default_mode = 'baryquark' if preset == 'baryquark_symmetric' else 'quarkyonic'
        request['momentum_mode'] = choose_option(
            'Momentum-space filling mode:',
            ['quarkyonic', 'baryquark'],
            default_key=default_mode,
        )
        default_lambda = 200.0 if needs_asymmetric else 300.0
        raw_lambda = input(f'Lambda momentum in MeV [{default_lambda}]: ').strip()
        request['lambda_momentum_mev'] = float(raw_lambda) if raw_lambda else default_lambda

    if needs_asymmetric:
        request['branch_mode'] = choose_option(
            'Asymmetric branch mode:',
            ['target_l', 'equal_b', 'both'],
            default_key='target_l',
            descriptions={
                'target_l': 'compute the b_pn != b_n branch',
                'equal_b': 'compute the b_pn = b_n branch',
                'both': 'generate and run both asymmetric branches',
            },
        )
        if guessed['neutron_star']:
            request['beta_equilibrium'] = True
            print('Beta equilibrium forced to True because neutron_star=True.')
        else:
            default_beta = preset != 'asymmetric_fixed_y'
            request['beta_equilibrium'] = parse_yes_no(
                input(f'Include beta equilibrium? [{"Y" if default_beta else "y/N"}]: '),
                default_beta,
            )
        raw_pf = input('Proton fractions as comma-separated values [0.0,0.1,0.2,0.3,0.4,0.5]: ')
        request['proton_fraction_values'] = parse_float_list(raw_pf, base['proton_fraction_values'])

    request['write_csv'] = parse_yes_no(input('Write CSV outputs? [Y/n]: '), True)
    request['write_json'] = parse_yes_no(input('Write JSON outputs? [Y/n]: '), True)
    default_plots = bool(python_info['has_matplotlib'])
    if not python_info['has_matplotlib']:
        print('Plots are disabled by default because the selected Python has no matplotlib installed.')
    request['write_plots'] = parse_yes_no(
        input(f'Write plots? [{"Y/n" if default_plots else "y/N"}]: '),
        default_plots,
    )
    request['plot_formats'] = ['png'] if request['write_plots'] else []
    return request


In [4]:
request = prompt_user_request(MODELS, MODEL_FAMILIES, PYTHON_INFO)
request


What do you want to simulate?
  1. ground_state_only - Fit symmetric nuclear matter at saturation only.
  2. critical_point - Ground state plus the finite-temperature critical point.
  3. symmetric_quarkyonic - Ground state plus symmetric quarkyonic matter.
  4. full_symmetric - Ground state, critical point, hadronic EOS, and symmetric quarkyonic matter.
  5. asymmetric_fixed_y - Asymmetric fixed-y quarkyonic matter without beta equilibrium.
  6. asymmetric_beta - Asymmetric quarkyonic matter in beta equilibrium.
  7. asymmetric_beta_neutron_star - Asymmetric beta-equilibrium matter plus a neutron-star sequence.
  8. baryquark_symmetric - Symmetric baryquark matter demo.
  9. custom - Choose each workflow block manually.
Which mean field do you want to use?
  1. clausius
  2. dieterici
  3. pr
  4. rks
  5. vdw
Which excluded-volume rule do you want to use?
  1. vdw
Using QMM model: rks
Momentum-space filling mode:
  1. quarkyonic
  2. baryquark


{'run_name': 'cs_test',
 'mean_field': 'rks',
 'excluded_volume': 'vdw',
 'model_name': 'rks',
 'preset': 'full_symmetric',
 'parameter_mode': 'none',
 'parameter_value': None,
 'target_k0': None,
 'quick_mode': True,
 'momentum_mode': 'quarkyonic',
 'lambda_momentum_mev': 305.0,
 'branch_mode': 'target_l',
 'beta_equilibrium': True,
 'proton_fraction_values': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
 'write_json': True,
 'write_csv': True,
 'write_plots': True,
 'plot_formats': ['png'],
 'output_subdir': None,
 'custom_workflows': None}

In [ ]:
# Alternative: edit this dictionary manually instead of using the prompt cell above.
# If you run the prompt cell, that `request` will be used automatically.
manual_request = default_request()
manual_request.update(
    {
        'run_name': 'manual_cs_ground_state',
        'mean_field': 'vdw',
        'excluded_volume': 'cs',
        'model_name': 'cs',
        'preset': 'ground_state_only',
        'quick_mode': True,
        'write_plots': False,
        'plot_formats': [],
    }
)
manual_request


In [5]:
request_to_use = request if 'request' in globals() and request else manual_request
print('Using request:')
print(json.dumps(request_to_use, indent=2))

configs = build_configs(ROOT, request_to_use, PYTHON_BIN)
print(f'\nGenerated {len(configs)} config(s):')
for index, config in enumerate(configs, 1):
    print(f'\nConfig {index}:')
    print(json.dumps(config, indent=2))

config_paths = save_configs(ROOT, configs)
for config_path in config_paths:
    print(f'\nSaved config to: {config_path}')


Using request:
{
  "run_name": "cs_test",
  "mean_field": "rks",
  "excluded_volume": "vdw",
  "model_name": "rks",
  "preset": "full_symmetric",
  "parameter_mode": "none",
  "parameter_value": null,
  "target_k0": null,
  "quick_mode": true,
  "momentum_mode": "quarkyonic",
  "lambda_momentum_mev": 305.0,
  "branch_mode": "target_l",
  "beta_equilibrium": true,
  "proton_fraction_values": [
    0.0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5
  ],
  "write_json": true,
  "write_csv": true,
  "write_plots": true,
  "plot_formats": [
    "png"
  ],
  "output_subdir": null,
  "custom_workflows": null
}

Generated config:
{
  "run_name": "cs_test",
  "model": {
    "name": "rks",
    "parameter_value": null,
    "parameter_search": {
      "enabled": false,
      "target_k0": null,
      "parameter_min": null,
      "parameter_max": null,
      "scan_steps": 121,
      "tol": 1e-08
    },
    "mean_field": "rks",
    "excluded_volume": "vdw"
  },
  "workflows": {
    "ground_state": true

In [6]:
run_results = run_configs(ROOT, config_paths, PYTHON_BIN)
for run_result in run_results:
    print('Command used:')
    print(run_result['command'])
    print()
    print(summarize_run(run_result['summary']))
    print()


Command used:
/opt/anaconda3/envs/tovsolver/bin/python -m qmm examples/generated/cs_test.json

run_name: cs_test
model: rks
summary_json: results/generated/cs_test/cs_test_summary.json
collection_quantum_summary_csv: results/generated/collection_quantum_summary.csv
collection_quantum_summary_png: results/generated/collection_quantum_summary.png
cs_test_ground_state_eos_png: results/generated/cs_test/cs_test_ground_state_eos.png
cs_test_quantum_diagnostics_png: results/generated/cs_test/cs_test_quantum_diagnostics.png
cs_test_symmetric_quarkyonic_png: results/generated/cs_test/cs_test_symmetric_quarkyonic.png
ground_state_csv: results/generated/cs_test/cs_test_ground_state.csv
hadronic_eos_csv: results/generated/cs_test/cs_test_hadronic_eos.csv
quantum_critical_csv: results/generated/cs_test/cs_test_quantum_critical.csv
symmetric_quarkyonic_csv: results/generated/cs_test/cs_test_symmetric_quarkyonic_curve.csv


In [7]:
for config in configs:
    output_dir = ROOT / config['output']['directory']
    print(f'Output directory: {output_dir}')
    for path in sorted(output_dir.glob('*')):
        print(path.relative_to(ROOT))
    print()


Output directory: /Users/dajuarez4/Documents/QuarkMatt/QMM/results/generated/cs_test
results/generated/cs_test/cs_test_ground_state.csv
results/generated/cs_test/cs_test_ground_state_eos.png
results/generated/cs_test/cs_test_hadronic_eos.csv
results/generated/cs_test/cs_test_quantum_critical.csv
results/generated/cs_test/cs_test_quantum_diagnostics.png
results/generated/cs_test/cs_test_summary.json
results/generated/cs_test/cs_test_symmetric_quarkyonic.png
results/generated/cs_test/cs_test_symmetric_quarkyonic_curve.csv
